In [1]:
## config
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("healthcare.duckdb")

con.execute("PRAGMA threads=8")
con.execute("SET memory_limit='6GB'")

import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

fm.fontManager.addfont(font_path)

font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print(font_name)

Noto Sans CJK JP


In [2]:
con.execute("show tables;").df()

,name
0,analysis_years
1,atc234_sick
2,atc3_sick
3,atc4_disease_breadth
4,atc4_disease_contribution
5,atc4_disease_growth
6,atc4_disease_growth_rank
7,atc4_disease_hhi
8,atc4_disease_prior
9,atc4_disease_top3


In [12]:
sqlList = []

sqlList.append(
    """CREATE OR REPLACE VIEW qa_basic AS

SELECT
    'mefi_sick' AS dataset,
    MIN(diagYm) AS min_diagYm,
    MAX(diagYm) AS max_diagYm,
    COUNT(DISTINCT diagYm) AS month_count,
    COUNT(DISTINCT insupTpCd) AS insup_type_count
FROM mefi_sick

UNION ALL

SELECT
    'atc3_sick',
    MIN(diagYm),
    MAX(diagYm),
    COUNT(DISTINCT diagYm),
    COUNT(DISTINCT insupTpCd)
FROM atc3_sick

UNION ALL

SELECT
    'atc4_sick',
    MIN(diagYm),
    MAX(diagYm),
    COUNT(DISTINCT diagYm),
    COUNT(DISTINCT insupTpCd)
FROM atc4_sick

UNION ALL

SELECT
    'cmpn_sick',
    MIN(diagYm),
    MAX(diagYm),
    COUNT(DISTINCT diagYm),
    COUNT(DISTINCT insupTpCd)
FROM cmpn_sick

UNION ALL

SELECT
    'atc4_region',
    MIN(diagYm),
    MAX(diagYm),
    COUNT(DISTINCT diagYm),
    COUNT(DISTINCT insupTpCd)
FROM atc4_region

UNION ALL

SELECT
    'cmpn_region',
    MIN(diagYm),
    MAX(diagYm),
    COUNT(DISTINCT diagYm),
    COUNT(DISTINCT insupTpCd)
FROM cmpn_region;""")


In [8]:
con.execute("""
SELECT DISTINCT insupTpCd
FROM mefi_sick
ORDER BY insupTpCd;

SELECT DISTINCT insupTpCd
FROM atc3_sick
ORDER BY insupTpCd;

SELECT DISTINCT insupTpCd
FROM atc4_sick
ORDER BY insupTpCd;

SELECT DISTINCT insupTpCd
FROM atc4_region
ORDER BY insupTpCd;
""").df()

,insupTpCd
0,4
1,5
2,7


In [13]:
sqlList.append("""
CREATE OR REPLACE VIEW qa_coverage_loss AS

WITH

mefi AS (
    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
        SUM(CAST(totUseQty AS DOUBLE)) AS qty,
        SUM(CAST(msupUseAmt AS DOUBLE)) AS amt
    FROM mefi_sick
    WHERE insupTpCd IN ('4','5','7')
    GROUP BY 1
),

atc3 AS (
    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
        SUM(CAST(totUseQty AS DOUBLE)) AS qty,
        SUM(CAST(msupUseAmt AS DOUBLE)) AS amt
    FROM atc3_sick
    WHERE insupTpCd IN ('4','5','7')
    GROUP BY 1
),

atc4 AS (
    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
        SUM(CAST(totUseQty AS DOUBLE)) AS qty,
        SUM(CAST(msupUseAmt AS DOUBLE)) AS amt
    FROM atc4_sick
    WHERE insupTpCd IN ('4','5','7')
    GROUP BY 1
),

cmpn AS (
    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
        SUM(CAST(totUseQty AS DOUBLE)) AS qty,
        SUM(CAST(msupUseAmt AS DOUBLE)) AS amt
    FROM cmpn_sick
    WHERE insupTpCd IN ('4','5','7')
    GROUP BY 1
)

SELECT
    m.year,

    m.amt AS mefi_amt,
    a3.amt AS atc3_amt,
    a4.amt AS atc4_amt,
    c.amt AS cmpn_amt,

    a3.amt / NULLIF(m.amt,0) AS atc3_vs_mefi_coverage,
    a4.amt / NULLIF(m.amt,0) AS atc4_vs_mefi_coverage,
    c.amt  / NULLIF(m.amt,0) AS cmpn_vs_mefi_coverage,

    a3.amt / NULLIF(m.amt,0) - 1 AS atc3_vs_mefi_loss,
    a4.amt / NULLIF(m.amt,0) - 1 AS atc4_vs_mefi_loss,
    c.amt  / NULLIF(m.amt,0) - 1 AS cmpn_vs_mefi_loss,

    m.qty AS mefi_qty,
    a3.qty AS atc3_qty,
    a4.qty AS atc4_qty,
    c.qty AS cmpn_qty,

    a3.qty / NULLIF(m.qty,0) AS atc3_qty_coverage,
    a4.qty / NULLIF(m.qty,0) AS atc4_qty_coverage,
    c.qty  / NULLIF(m.qty,0) AS cmpn_qty_coverage

FROM mefi m
LEFT JOIN atc3 a3 USING(year)
LEFT JOIN atc4 a4 USING(year)
LEFT JOIN cmpn c USING(year)
ORDER BY year;
""")

In [14]:
sqlList.append("""
CREATE OR REPLACE VIEW qa_atc4_coverage AS

WITH mefi AS (

    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,

        SUM(CAST(msupUseAmt AS DOUBLE)) AS mefi_amt

    FROM mefi_sick

    WHERE insupTpCd IN ('4','5','7')

    GROUP BY 1
),

atc4 AS (

    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,

        atcStep4Cd,
        MAX(atcStep4CdNm) AS atc4_name,

        SUM(CAST(msupUseAmt AS DOUBLE)) AS atc4_amt

    FROM atc4_sick

    WHERE insupTpCd IN ('4','5','7')
      AND atcStep4Cd IS NOT NULL

    GROUP BY
        1,2
)

SELECT
    a.year,
    a.atcStep4Cd,
    a.atc4_name,
    a.atc4_amt,
    m.mefi_amt,

    a.atc4_amt / NULLIF(m.mefi_amt,0)
        AS atc4_vs_national_mefi_ratio

FROM atc4 a

CROSS JOIN mefi m

WHERE a.year = m.year;
""")

In [10]:
con.execute("""
SELECT
    'atc4_sick' AS dataset,
    COUNT(*) FILTER (WHERE msupUseAmt IS NULL) AS null_amt,
    COUNT(*) FILTER (WHERE msupUseAmt < 0) AS negative_amt,
    COUNT(*) FILTER (WHERE totUseQty IS NULL) AS null_qty,
    COUNT(*) FILTER (WHERE totUseQty < 0) AS negative_qty
FROM atc4_sick

UNION ALL

SELECT
    'cmpn_sick',
    COUNT(*) FILTER (WHERE msupUseAmt IS NULL),
    COUNT(*) FILTER (WHERE msupUseAmt < 0),
    COUNT(*) FILTER (WHERE totUseQty IS NULL),
    COUNT(*) FILTER (WHERE totUseQty < 0)
FROM cmpn_sick;
""").df()

,dataset,null_amt,negative_amt,null_qty,negative_qty
0,atc4_sick,0,0,0,0
1,cmpn_sick,0,0,0,0


In [15]:
for isql in sqlList[1:]:
    con.execute(isql)

In [16]:
display(con.execute("SELECT * FROM qa_basic;").df())
display(con.execute("SELECT * FROM qa_coverage_loss;").df())
display(con.execute("SELECT * FROM qa_atc4_coverage;").df())


,dataset,min_diagYm,max_diagYm,month_count,insup_type_count
0,mefi_sick,202001,202212,36,3
1,atc3_sick,202001,202212,36,3
2,atc4_sick,202001,202212,36,3
3,cmpn_sick,202001,202212,36,3
4,atc4_region,202001,202212,36,3
5,cmpn_region,202001,202212,36,3


,year,mefi_amt,atc3_amt,atc4_amt,cmpn_amt,atc3_vs_mefi_coverage,atc4_vs_mefi_coverage,cmpn_vs_mefi_coverage,atc3_vs_mefi_loss,atc4_vs_mefi_loss,cmpn_vs_mefi_loss,mefi_qty,atc3_qty,atc4_qty,cmpn_qty,atc3_qty_coverage,atc4_qty_coverage,cmpn_qty_coverage
0,2020,3.784598e+13,3.750884e+13,3.570061e+13,1.967897e+13,0.991092,0.943313,0.519975,-0.008908,-0.056687,-0.480025,8.208685e+10,8.067634e+10,7.616320e+10,5.242156e+10,0.982817,0.927837,0.638611
1,2021,4.125671e+13,4.082782e+13,3.864317e+13,2.161961e+13,0.989604,0.936652,0.524027,-0.010396,-0.063348,-0.475973,8.723620e+10,8.561065e+10,8.143527e+10,5.574549e+10,0.981366,0.933503,0.639018
2,2022,4.215731e+13,4.170131e+13,3.864888e+13,2.326689e+13,0.989183,0.916778,0.551906,-0.010817,-0.083222,-0.448094,9.853967e+10,9.682942e+10,9.263916e+10,6.615725e+10,0.982644,0.940120,0.671377


,year,atcStep4Cd,atc4_name,atc4_amt,mefi_amt,atc4_vs_national_mefi_ratio
0,2020,A10BG,Thiazolidinediones,1.102682e+11,3.784598e+13,0.002914
1,2020,A11GA,"Ascorbic acid (vitamin C), plain",1.449661e+09,3.784598e+13,0.000038
2,2020,B05AA,Blood substitutes and plasma protein fractions,1.260880e+11,3.784598e+13,0.003332
3,2020,C07AG,Alpha and beta blocking agents,1.483769e+11,3.784598e+13,0.003921
4,2020,C09DA,Angiotensin II receptor blockers (ARBs) and di...,3.068616e+11,3.784598e+13,0.008108
...,...,...,...,...,...,...
902,2022,D06BX,Other chemotherapeutics,1.515842e+08,4.215731e+13,0.000004
903,2022,A16AA,Amino acids and derivatives,5.363784e+07,4.215731e+13,0.000001
904,2020,J01EE,"Combinations of sulfonamides and trimethoprim,...",1.642469e+08,3.784598e+13,0.000004
905,2021,N05AD,Butyrophenone derivatives,3.087435e+08,4.125671e+13,0.000007


In [ ]:
display(con.execute("SELECT * FROM qa_atc4_coverage order by atc4_vs_national_mefi_ratio desc limit 10;").df())

,year,atcStep4Cd,atc4_name,atc4_amt,mefi_amt,atc4_vs_national_mefi_ratio
0,2020,C10AA,HMG CoA reductase inhibitors,1.683432e+12,3.784598e+13,0.044481
1,2021,C10AA,HMG CoA reductase inhibitors,1.790090e+12,4.125671e+13,0.043389
2,2022,C10AA,HMG CoA reductase inhibitors,1.753111e+12,4.215731e+13,0.041585
3,2022,A02BC,Proton pump inhibitors,1.466112e+12,4.215731e+13,0.034777
4,2022,C10BA,Combinations of various lipid modifying agents,1.444571e+12,4.215731e+13,0.034266
5,2021,B01AC,Platelet aggregation inhibitors excl. heparin,1.385542e+12,4.125671e+13,0.033583
6,2021,A02BC,Proton pump inhibitors,1.383735e+12,4.125671e+13,0.033540
7,2020,B01AC,Platelet aggregation inhibitors excl. heparin,1.262555e+12,3.784598e+13,0.033360
8,2020,L01XC,Monoclonal antibodies,1.237211e+12,3.784598e+13,0.032691
9,2022,B01AC,Platelet aggregation inhibitors excl. heparin,1.366683e+12,4.215731e+13,0.032419


: 

In [17]:
con.execute("""
WITH atc4_base AS
(
SELECT
    CAST(LEFT(CAST(diagYm AS VARCHAR),4) AS INTEGER) AS year,
    CAST(diagYm AS VARCHAR) AS diagYm,

    atcStep4Cd,
    atcStep4CdNm,

    st3SickSym,
    st3SickSymNm,

    insupTpCd,

    CAST(totUseQty AS DOUBLE) AS use_qty,
    CAST(msupUseAmt AS DOUBLE) AS use_amt

FROM atc4_sick

WHERE insupTpCd IN ('4','5','7')
  AND atcStep4Cd IS NOT NULL   
)
SELECT
    year,
    COUNT(*) AS rows,
    COUNT(*) FILTER (
        WHERE atcStep4Cd IS NULL
    ) AS null_atc4
FROM atc4_base
GROUP BY year
ORDER BY year;
""").df()

,year,rows,null_atc4
0,2020,8422000,0
1,2021,8521968,0
2,2022,8430190,0


In [9]:
con.execute("""
-- 절대적인 임계값을 임의로 정하지 않고 Quartile을 사용한다.
CREATE OR REPLACE VIEW atc4_market_position AS
SELECT
    g.*,
    p.positive_growth_periods,

    NTILE(4) OVER (
        ORDER BY latest_amt
    ) AS size_quartile,

    NTILE(4) OVER (
        ORDER BY cumulative_growth_rate
    ) AS growth_quartile,

    NTILE(4) OVER (
        ORDER BY positive_growth_periods
    ) AS persistence_quartile
FROM atc4_market_growth g 
LEFT JOIN atc4_persistence p 
ON g.atcStep4Cd = p.atcStep4Cd;
""")
# 임계값을 Quartile로 정하는 이유
# Median: 큰/작은 시장의 2분할
# Quartile: 시장을 4개 포지션으로 구분
# Quintile: 상위 20% 후보 선정에 적합하지만 작은 표본에서 과도한 정밀성을 줄 수 있음

# 따라서 기본 분석 = Quartile,
# 민감도 분석 = Median / Quintile로 한다.

con.execute("""
CREATE OR REPLACE VIEW atc4_market_share AS
WITH yearly AS (
    SELECT
        year,
        atcStep4Cd,
        atcStep4CdNm,
        use_amt,

        SUM(use_amt) OVER (
            PARTITION BY year
        ) AS total_market
    FROM atc4_market_yearly
),

share AS (
    SELECT
        *,
        use_amt / NULLIF(total_market, 0) AS market_share
    FROM yearly
)

SELECT
    atcStep4Cd,
    atcStep4CdNm,

    MAX(
        CASE
            WHEN year = (SELECT base_year FROM analysis_years)
            THEN market_share
        END
    ) AS base_share,

    MAX(
        CASE
            WHEN year = (SELECT latest_year FROM analysis_years)
            THEN market_share
        END
    ) AS latest_share

FROM share
GROUP BY
    1,2;

-- 최종 변화:

CREATE OR REPLACE VIEW atc4_share_change AS
SELECT
    *,
    latest_share - base_share AS share_change
FROM atc4_market_share;
""")

In [13]:
con.execute("""
CREATE OR REPLACE VIEW atc4_disease_yearly aS 
SELECT 
CAST(LEFT(CAST(diagYm AS VARCHAR), 4) AS INTEGER) AS year, 
atcStep4Cd, atcStep4CdNm, st3SickSym, st3SickSymNm, 
SUM(msupUseAmt) AS use_amt FROM atc4_sick 
GROUP BY
1,2,3,4,5
;

CREATE OR REPLACE VIEW atc4_disease_growth AS 
WITH x AS ( 
SELECT 
d.atcStep4Cd, d.atcStep4CdNm, d.st3SickSym, d.st3SickSymNm, 
SUM( CASE WHEN d.year = a.base_year THEN d.use_amt ELSE 0 END ) AS base_amt, 
SUM( CASE WHEN d.year = a.prev_year THEN d.use_amt ELSE 0 END ) AS prev_amt, 
SUM( CASE WHEN d.year = a.latest_year THEN d.use_amt ELSE 0 END ) AS latest_amt 
FROM atc4_disease_yearly d 
cROSS JOIN analysis_years a 
GROUP BY 
1,2,3,4 
) 
SELECT *, latest_amt - base_amt AS growth_amt, 
(latest_amt - base_amt) / NULLIF(base_amt, 0) AS cumulative_growth_rate, 
POWER( latest_amt / NULLIF(base_amt, 0), 1.0 / 2 ) - 1 AS cagr 
FROM x 
WHERE base_amt > 0
;
CREATE OR REPLACE VIEW atc4_disease_contribution AS 
SELECT *, SUM(growth_amt) OVER ( PARTITION BY atcStep4Cd ) AS atc4_growth_amt, 
growth_amt / NULLIF( SUM(growth_amt) OVER ( PARTITION BY atcStep4Cd ), 0 ) AS growth_contribution 
FROM atc4_disease_growth;


-- 상병 성장집중도
CREATE OR REPLACE VIEW atc4_disease_growth_rank AS
SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY atcStep4Cd
        ORDER BY growth_amt DESC
    ) AS growth_rank

FROM atc4_disease_contribution;

-- Top 3 성장기여:

CREATE OR REPLACE VIEW atc4_disease_top3 AS
SELECT
    atcStep4Cd,
    atcStep4CdNm,

    SUM(
        CASE
            WHEN growth_rank <= 3
            THEN growth_contribution
            ELSE 0
        END
    ) AS top3_growth_contribution

FROM atc4_disease_growth_rank

GROUP BY
    1,2
    ;

-- 성장폭
CREATE OR REPLACE VIEW atc4_disease_breadth AS 
SELECT atcStep4Cd, atcStep4CdNm, 
COUNT(*) FILTER ( WHERE growth_amt > 0 ) AS growing_disease_count, 
COUNT(*) FILTER ( WHERE growth_amt < 0 ) AS declining_disease_count, 
COUNT(*) AS total_disease_count, COUNT(*) FILTER ( WHERE growth_amt > 0 ) * 1.0 / NULLIF(COUNT(*), 0) AS growth_breadth 
FROM atc4_disease_growth 
GROUP BY 1,2;


""")

con.execute("""
-- atc4의 상병사전분포도 // 최근 연도 기준으로 ATC4 내부에서 각 상병이 차지하는 비중이다.

CREATE OR REPLACE VIEW atc4_disease_prior AS
SELECT
    atcStep4Cd,
    atcStep4CdNm,
    st3SickSym,
    st3SickSymNm,

    latest_amt,

    latest_amt /
        NULLIF(
            SUM(latest_amt) OVER (
                PARTITION BY atcStep4Cd
            ),
            0
        ) AS disease_share,

    growth_amt,
    cumulative_growth_rate,
    cagr

FROM atc4_disease_growth
;

CREATE OR REPLACE VIEW atc4_disease_hhi AS 
SELECT 
atcStep4Cd, atcStep4CdNm, SUM( POWER(disease_share, 2) ) AS disease_hhi 
FROM atc4_disease_prior 
GROUP BY 1,2
;

""")



In [15]:
con.execute("""
CREATE OR REPLACE VIEW atc4_market_opportunity AS 
SELECT p.atcStep4Cd, p.atcStep4CdNm, p.base_amt, p.prev_amt, p.latest_amt, 
p.growth_amt, p.cumulative_growth_rate, p.cagr, 
p.size_quartile, p.growth_quartile, p.persistence_quartile, 
ps.positive_growth_periods, sc.base_share, sc.latest_share, sc.share_change, 
b.growing_disease_count, b.declining_disease_count, b.growth_breadth,
t.top3_growth_contribution, h.disease_hhi 
FROM atc4_market_position p 
LEFT JOIN atc4_persistence ps ON p.atcStep4Cd = ps.atcStep4Cd 
LEFT JOIN atc4_share_change sc ON p.atcStep4Cd = sc.atcStep4Cd 
LEFT JOIN atc4_disease_breadth b ON p.atcStep4Cd = b.atcStep4Cd 
LEFT JOIN atc4_disease_top3 t ON p.atcStep4Cd = t.atcStep4Cd 
LEFT JOIN atc4_disease_hhi h ON p.atcStep4Cd = h.atcStep4Cd
;
CREATE OR REPLACE VIEW atc4_market_segment AS 
SELECT *, 
CASE 
WHEN size_quartile = 4 AND growth_quartile = 4 AND persistence_quartile = 4 
THEN 'Core Growth Market' 
WHEN growth_quartile = 4 AND size_quartile IN (2,3) 
THEN 'Emerging Growth Market' 
WHEN growth_quartile = 1 AND persistence_quartile = 1 
THEN 'Declining Market' ELSE 'Other Market' END AS market_segment 
FROM atc4_market_opportunity
;
""")